# Steering di un concetto su Llama-3.1-8B-Instruct

Progetto d'esame: inietto una direzione ("concetto") nelle attivazioni del modello
(**Activation Addition**) e osservo quando il concetto compare nelle risposte.

I prompt stanno in un file separato **`prompts.py`** (così il notebook resta leggibile):
va caricato una volta nella sezione 3.

**Tre test:**
1. **Configurazioni** — provo diversi *layer* e diverse *intensità* di iniezione, su
   prompt *trigger* (dove il concetto dovrebbe comparire) e *non-trigger* (dove non dovrebbe).
2. **Scelta della migliore** — la configurazione migliore è coerente, fa comparire il
   concetto sui trigger e NON lo fa comparire sui non-trigger.
3. **Con vs senza steering** — confronto il modello con steering (config. migliore) e il
   modello normale sugli stessi prompt.

Ogni test salva i suoi risultati (testo **completo, non troncato**) e il suo grafico in `results/<Concetto>/`.

**Prima di iniziare:** `Runtime → Change runtime type → GPU`, e aggiungi `HF_TOKEN` nei Secrets (🔑).


## 1. Dipendenze

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes > /dev/null
print("ok")

## 2. Token Hugging Face (Llama è gated)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("token ok")
except Exception as e:
    from huggingface_hub import login
    login()

## 3. Prompt (dal file esterno `prompts.py`)
Se non è già presente, la cella chiede di caricarlo.

In [ ]:
import importlib
try:
    import prompts; importlib.reload(prompts)
except ModuleNotFoundError:
    from google.colab import files
    print("Carica il file prompts.py …")
    files.upload()
    import prompts; importlib.reload(prompts)
print("Concetti disponibili:", list(prompts.CONCEPTS))

## 4. Modello e funzioni dello steering
Il modello si carica in 4-bit (sta su una T4). Poche funzioni semplici:
`activation` (legge le attivazioni), `steering_vector` (costruisce la direzione del concetto),
`generate` (genera, eventualmente iniettando il vettore).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True,
                                           bnb_4bit_compute_dtype=torch.float16))
model.eval()
print("Modello caricato.  Numero di layer:", model.config.num_hidden_layers)


def _capture(layer):
    """Piccolo hook che salva l'attivazione in uscita da un layer."""
    store = {}
    def hook(m, i, o):
        store["h"] = (o[0] if isinstance(o, tuple) else o).detach()
    return store, hook


def activation(text, layer):
    """Attivazione media (mean-pool sui token) del layer dato per un testo."""
    store, hook = _capture(layer)
    h = model.model.layers[layer].register_forward_hook(hook)
    with torch.no_grad():
        model(**tokenizer(text, return_tensors="pt").to(model.device))
    h.remove()
    return store["h"][0].mean(0).float().cpu()


def steering_vector(pairs, layer):
    """Direzione del concetto = media delle differenze (frase con - frase senza)."""
    diffs = [activation(pos, layer) - activation(neg, layer) for pos, neg in pairs]
    v = torch.stack(diffs).mean(0)
    return v / v.norm()                      # vettore unitario


def norm_at(layer, s="The quick brown fox jumps over the lazy dog."):
    """Norma tipica delle attivazioni al layer: rende l'intensità confrontabile fra layer."""
    store, hook = _capture(layer)
    h = model.model.layers[layer].register_forward_hook(hook)
    with torch.no_grad():
        model(**tokenizer(s, return_tensors="pt").to(model.device))
    h.remove()
    return store["h"][0].float().norm(dim=-1).mean().item()


def generate(prompt, vector=None, strength=0.0, layer=0):
    """Genera una risposta. Se vector!=None inietta strength*vector nel layer (ActAdd)."""
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                         tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    handle = None
    if vector is not None and strength:
        add = (vector * strength).to(model.device, model.dtype)
        def hook(m, i, o):
            return (o[0] + add,) + o[1:] if isinstance(o, tuple) else o + add
        handle = model.model.layers[layer].register_forward_hook(hook)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=True,
                             temperature=0.7, top_p=0.9,
                             pad_token_id=tokenizer.eos_token_id)
    if handle:
        handle.remove()
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True).strip()


def mentions(text, aliases):
    """True se il concetto compare nel testo."""
    t = text.lower()
    return any(a in t for a in aliases)


def is_coherent(text):
    """Euristica semplice: la risposta non è una ripetizione degenerata."""
    w = text.split()
    if len(w) < 5:
        return True
    return max(w.count(x) for x in set(w)) / len(w) < 0.4

## 5. Scelta del concetto e delle configurazioni da testare
Cambia `CONCEPT` e, se vuoi, i layer/intensità da provare.

In [ ]:
CONCEPT = "Napoleone"        # "Napoleone" | "Colosseo" | "Apple"

d        = prompts.CONCEPTS[CONCEPT]
ALIASES  = d["aliases"]
PAIRS    = d["vector_pairs"]
TRIGGER  = d["trigger_prompts"]
NEUTRAL  = d["non_trigger_prompts"]

TEST_LAYERS      = [8, 12, 16]          # dove iniettare il concetto
TEST_INTENSITIES = [0.08, 0.12, 0.16]   # quanto (frazione della norma del layer)

import os, json
import numpy as np
import matplotlib.pyplot as plt

OUT = f"results/{CONCEPT}"
os.makedirs(OUT, exist_ok=True)
print(f"Concetto: {CONCEPT} | trigger: {len(TRIGGER)} | non-trigger: {len(NEUTRAL)}")
print(f"Griglia: {len(TEST_LAYERS)} layer x {len(TEST_INTENSITIES)} intensità")

## TEST 1 — Diverse configurazioni di steering
Per ogni (layer, intensità) genero le risposte ai prompt trigger e non-trigger e misuro
quante volte compare il concetto. Il testo **completo** finisce nel file; a schermo solo il riepilogo.

In [ ]:
# vettore e norma per ogni layer (calcolati una volta)
vectors = {L: steering_vector(PAIRS, L) for L in TEST_LAYERS}
norms   = {L: norm_at(L) for L in TEST_LAYERS}

rows1 = []
trig_grid = np.zeros((len(TEST_LAYERS), len(TEST_INTENSITIES)))
neut_grid = np.zeros((len(TEST_LAYERS), len(TEST_INTENSITIES)))
coh_grid  = np.zeros((len(TEST_LAYERS), len(TEST_INTENSITIES)))

for i, L in enumerate(TEST_LAYERS):
    for j, frac in enumerate(TEST_INTENSITIES):
        strength = frac * norms[L]
        t_hits = n_hits = coh_ok = 0
        for kind, plist in [("trigger", TRIGGER), ("non-trigger", NEUTRAL)]:
            for p in plist:
                r = generate(p, vectors[L], strength, L)
                m, c = mentions(r, ALIASES), is_coherent(r)
                t_hits += int(kind == "trigger" and m)
                n_hits += int(kind == "non-trigger" and m)
                coh_ok += int(c)
                rows1.append({"layer": L, "intensity": frac, "kind": kind,
                              "prompt": p, "response": r, "mentions": m, "coherent": c})
        trig_grid[i, j] = t_hits / len(TRIGGER)
        neut_grid[i, j] = n_hits / len(NEUTRAL)
        coh_grid[i, j]  = coh_ok / (len(TRIGGER) + len(NEUTRAL))
        print(f"L{L:>2} int {frac:.0%} | trigger {trig_grid[i,j]:.0%} | "
              f"non-trigger {neut_grid[i,j]:.0%} | coerenza {coh_grid[i,j]:.0%}")

# --- salvataggio testo COMPLETO (nessun troncamento) ---
with open(f"{OUT}/test1_configurations.txt", "w") as f:
    f.write(f"TEST 1 — {CONCEPT}: configurazioni di steering\n{'='*80}\n")
    for r in rows1:
        f.write(f"\n[layer {r['layer']} | intensità {r['intensity']:.0%} | {r['kind']}] "
                f"mentions={r['mentions']} coherent={r['coherent']}\n")
        f.write(f"PROMPT   : {r['prompt']}\n")
        f.write(f"RESPONSE : {r['response']}\n")
        f.write("-" * 80 + "\n")
json.dump(rows1, open(f"{OUT}/test1_configurations.json", "w"), indent=2)

# --- grafico: due heatmap (trigger vogliamo alto, non-trigger vogliamo basso) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, grid, title in [(axes[0], trig_grid, "Trigger  (meglio ALTO)"),
                        (axes[1], neut_grid, "Non-trigger  (meglio BASSO)")]:
    im = ax.imshow(grid, cmap="YlOrRd", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(TEST_INTENSITIES)))
    ax.set_xticklabels([f"{f:.0%}" for f in TEST_INTENSITIES])
    ax.set_yticks(range(len(TEST_LAYERS)))
    ax.set_yticklabels([f"L{L}" for L in TEST_LAYERS])
    ax.set_xlabel("intensità"); ax.set_ylabel("layer"); ax.set_title(title)
    for i in range(len(TEST_LAYERS)):
        for j in range(len(TEST_INTENSITIES)):
            ax.text(j, i, f"{grid[i,j]:.0%}", ha="center", va="center")
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle(f"TEST 1 — {CONCEPT}: mention rate per configurazione")
plt.tight_layout()
plt.savefig(f"{OUT}/test1_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvati: test1_configurations.txt / .json e test1_heatmap.png")

## TEST 2 — Confronto e scelta della configurazione migliore
Migliore = coerente, alta sui trigger, bassa sui non-trigger.
Punteggio semplice: `coerenza × (trigger − non_trigger)`.

In [ ]:
score_grid = coh_grid * (trig_grid - neut_grid)

ranking = []
for i, L in enumerate(TEST_LAYERS):
    for j, frac in enumerate(TEST_INTENSITIES):
        ranking.append({"layer": L, "intensity": frac,
                        "trigger": float(trig_grid[i, j]),
                        "non_trigger": float(neut_grid[i, j]),
                        "coherence": float(coh_grid[i, j]),
                        "score": float(score_grid[i, j])})
ranking.sort(key=lambda r: r["score"], reverse=True)
best = ranking[0]
print(f"MIGLIORE → layer {best['layer']}, intensità {best['intensity']:.0%} | "
      f"trigger {best['trigger']:.0%}, non-trigger {best['non_trigger']:.0%}, "
      f"coerenza {best['coherence']:.0%}")

with open(f"{OUT}/test2_best_config.txt", "w") as f:
    f.write(f"TEST 2 — {CONCEPT}: scelta della configurazione\n{'='*60}\n")
    f.write(f"MIGLIORE: layer {best['layer']} | intensità {best['intensity']:.0%}\n\n")
    f.write(f"{'layer':>6}{'intens':>9}{'trigger':>10}{'non-trig':>10}{'coer':>8}{'score':>8}\n")
    for r in ranking:
        f.write(f"{r['layer']:>6}{r['intensity']:>8.0%}{r['trigger']:>10.0%}"
                f"{r['non_trigger']:>10.0%}{r['coherence']:>8.0%}{r['score']:>8.2f}\n")

# --- grafico: ogni config come punto (x = non-trigger, y = trigger). Ideale: alto-sinistra ---
plt.figure(figsize=(6, 6))
for r in ranking:
    plt.scatter(r["non_trigger"], r["trigger"], s=60 + 160 * r["coherence"],
                color="#457b9d", alpha=0.7)
    plt.annotate(f"L{r['layer']}·{r['intensity']:.0%}",
                 (r["non_trigger"], r["trigger"]),
                 textcoords="offset points", xytext=(6, 4), fontsize=8)
plt.scatter(best["non_trigger"], best["trigger"], marker="*", s=450,
            color="gold", edgecolor="black", zorder=5, label="migliore")
plt.xlabel("mention non-trigger  (meglio ←)")
plt.ylabel("mention trigger  (meglio ↑)")
plt.xlim(-0.05, 1.05); plt.ylim(-0.05, 1.05)
plt.title(f"TEST 2 — {CONCEPT}: scelta della configurazione\n(dimensione punto = coerenza)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUT}/test2_ranking.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvati: test2_best_config.txt e test2_ranking.png")

## TEST 3 — Modello con steering vs modello senza
Uso la configurazione migliore del Test 2 e confronto, sugli stessi prompt trigger e
non-trigger, il modello normale e quello con lo steering iniettato.

In [ ]:
BEST_L, BEST_FRAC = best["layer"], best["intensity"]
vec = steering_vector(PAIRS, BEST_L)
strength = BEST_FRAC * norm_at(BEST_L)
print(f"Configurazione migliore: layer {BEST_L}, intensità {BEST_FRAC:.0%}\n")

rows3 = []
count = {("base", "trigger"): 0, ("base", "non-trigger"): 0,
         ("steered", "trigger"): 0, ("steered", "non-trigger"): 0}

for kind, plist in [("trigger", TRIGGER), ("non-trigger", NEUTRAL)]:
    for p in plist:
        r_base  = generate(p)                            # senza steering
        r_steer = generate(p, vec, strength, BEST_L)     # con steering
        for cond, resp in [("base", r_base), ("steered", r_steer)]:
            m = mentions(resp, ALIASES)
            count[(cond, kind)] += int(m)
            rows3.append({"condition": cond, "kind": kind, "prompt": p,
                          "response": resp, "mentions": m})
        print(f"[{kind}] {p}")
        print(f"    senza steering: {'✅ compare' if mentions(r_base, ALIASES) else '❌ assente'}")
        print(f"    con steering  : {'✅ compare' if mentions(r_steer, ALIASES) else '❌ assente'}")

# --- salvataggio testo COMPLETO ---
with open(f"{OUT}/test3_comparison.txt", "w") as f:
    f.write(f"TEST 3 — {CONCEPT}: con vs senza steering "
            f"(layer {BEST_L}, intensità {BEST_FRAC:.0%})\n{'='*80}\n")
    for r in rows3:
        f.write(f"\n[{r['condition']} | {r['kind']}] mentions={r['mentions']}\n")
        f.write(f"PROMPT   : {r['prompt']}\n")
        f.write(f"RESPONSE : {r['response']}\n")
        f.write("-" * 80 + "\n")
json.dump(rows3, open(f"{OUT}/test3_comparison.json", "w"), indent=2)

# --- grafico a barre: base vs steered su trigger e non-trigger ---
labels = ["trigger", "non-trigger"]
sizes  = {"trigger": len(TRIGGER), "non-trigger": len(NEUTRAL)}
base_rates  = [count[("base", k)] / sizes[k] for k in labels]
steer_rates = [count[("steered", k)] / sizes[k] for k in labels]
x = np.arange(2); w = 0.35
plt.figure(figsize=(6, 5))
plt.bar(x - w/2, base_rates,  w, label="senza steering", color="#a8dadc")
plt.bar(x + w/2, steer_rates, w, label="con steering",   color="#e63946")
for xi, v in zip(x - w/2, base_rates):  plt.text(xi, v + 0.02, f"{v:.0%}", ha="center")
for xi, v in zip(x + w/2, steer_rates): plt.text(xi, v + 0.02, f"{v:.0%}", ha="center")
plt.xticks(x, labels); plt.ylim(0, 1.15); plt.ylabel("mention rate"); plt.legend()
plt.title(f"TEST 3 — {CONCEPT}: con vs senza steering (L{BEST_L}, {BEST_FRAC:.0%})")
plt.tight_layout()
plt.savefig(f"{OUT}/test3_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvati: test3_comparison.txt / .json e test3_comparison.png")

## (Opzionale) Scarica tutti i risultati

In [ ]:
!zip -rq results.zip results
from google.colab import files
files.download("results.zip")